# Other LLM Providers & Deployment Options

Beyond OpenAI, Gemini, Claude, Grok, and DeepSeek, production systems routinely encounter **Mistral**, **Meta Llama**, **Cohere**, **cloud marketplaces**, and **local inference**. This notebook surveys those options and patterns for **multi-provider** architectures.

For integrated hands-on code, this course focuses on OpenAI, Gemini, and Claude. The concepts here apply when you expand to additional vendors.


---

## 1. Why Look Beyond Three Providers?

| Reason | Example |
|--------|---------|
| **Cost optimization** | Route simple queries to Mistral or DeepSeek |
| **Compliance** | Azure OpenAI or Bedrock with private networking |
| **Open weights** | Self-host Llama for data sovereignty |
| **Specialization** | Cohere for embeddings and retrieval |
| **Resilience** | Failover if one API has an outage |
| **Regional availability** | Some models only on certain clouds |

No single vendor wins every dimension — mature teams often use **two or more** providers.


---

## 2. Mistral AI

**Mistral** (France) offers both **open-weight models** (e.g. Mistral, Mixtral) and a **hosted API**.

| Aspect | Details |
|--------|---------|
| **Strengths** | Efficient models, European presence, open-weight option |
| **API** | REST API with Mistral Python SDK |
| **Use cases** | Cost-sensitive EU deployments, on-prem Mixtral |
| **Access** | [console.mistral.ai](https://console.mistral.ai) |

Mistral is a common choice when teams want strong performance without US-only cloud dependency.


---

## 3. Meta Llama (Hosted & Self-Hosted)

**Meta** releases **Llama** open-weight models. You typically do **not** call Meta directly — you:

| Path | How |
|------|-----|
| **Self-host** | Ollama, vLLM, llama.cpp on your hardware |
| **Cloud marketplaces** | Bedrock, Vertex, Azure, Together, Groq |
| **Hugging Face** | Inference endpoints |

Llama excels when you need **control over weights** and can invest in GPU infrastructure.


---

## 4. Cohere

**Cohere** focuses on **enterprise NLP** — especially **embeddings**, **reranking**, and **retrieval** for RAG pipelines.

| Product | Use |
|---------|-----|
| Embed models | Vector search |
| Rerank | Improve retrieval quality |
| Command | Chat / generation |

Often used **alongside** OpenAI or Claude — Cohere for retrieval, frontier model for final answer synthesis.


---

## 5. Cloud Marketplaces

### 5.1 Azure OpenAI Service

- OpenAI models on Microsoft Azure
- Azure AD, private endpoints, regional residency
- Same `openai` SDK with Azure `base_url` and deployment names

### 5.2 AWS Bedrock

- Multiple vendors (Anthropic Claude, Meta Llama, Mistral, Amazon Titan)
- Single AWS API with **IAM** authentication
- Unified billing through AWS

### 5.3 Google Vertex AI

- Enterprise Gemini on GCP
- IAM, VPC-SC, enterprise SLAs
- Compared to simpler Google AI Studio API keys

**Pattern:** You trade API-key simplicity for cloud **identity, networking, and compliance** controls.


---

## 6. Local Inference (Ollama)

**Ollama** runs open-weight models locally and exposes an **OpenAI-compatible** endpoint:

```python
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # required but ignored locally
)

response = client.chat.completions.create(
    model="llama3",
    messages=[{"role": "user", "content": "Hello!"}],
)
print(response.choices[0].message.content)
```

| Pros | Cons |
|------|------|
| Free inference (after hardware) | Weaker models than frontier APIs |
| Full privacy | You manage RAM/VRAM and updates |
| Offline development | No provider-managed scaling |

Ideal for **learning**, **prototyping**, and **privacy-sensitive** dev environments.


---

## 7. Multi-Provider Architecture

Production apps often implement a **router**:

```
User request
     |
     v
+------------+
|  Router    |  -- simple prompt --> cheap model (mini, Haiku, Flash)
+------------+
     |
     +-- complex / high stakes --> frontier model (GPT-4o, Opus, Pro)
     |
     +-- provider A down ---------> fallback to provider B
```

**Design tips:**

- Normalize on an **OpenAI-compatible interface** where possible (OpenAI, xAI, DeepSeek, Ollama)
- Keep **provider-specific** adapters for Gemini and Claude Messages API
- Log **latency, cost, and quality** per provider to inform routing rules
- Store API keys in **secrets managers**, not source code


---

## 8. Provider Comparison (Extended)

| Provider | API style | Best known for |
|----------|-----------|----------------|
| OpenAI | Chat completions (reference) | Ecosystem, tools, embeddings |
| Gemini | `generate_content` | Multimodal, Google Cloud |
| Claude | Messages API | Long docs, careful reasoning |
| Grok (xAI) | OpenAI-compatible | X/real-time angle |
| DeepSeek | OpenAI-compatible | Cost, R1 reasoning, open weights |
| Mistral | Mistral API | EU, efficient open weights |
| Cohere | Cohere API | Embeddings, rerank, RAG |
| Ollama | OpenAI-compatible local | Privacy, offline dev |


---

## 9. Summary

The LLM provider space extends far beyond three names. **Cloud marketplaces** add compliance; **open weights** add control; **routers** add resilience and cost savings. Start with one provider to learn, then expand deliberately based on measured cost, quality, and policy requirements.
